# PI-CAI Domain Adaptation on Kaggle

This notebook trains the Z-SSMNet model natively on Kaggle using Kaggle Datasets.
It is configured for **Domain Adaptation**, meaning you can train on 2 centers (e.g. RUMC and ZGT) while completely holding out the 3rd center (PCNN) for testing later.

### Prerequisites:
1. Ensure you have added your `picai-pre-processed-dataset` to this notebook (via the **Add Data** button on the right).
2. Ensure you have turned on **Internet Access** in the Kaggle notebook settings.
3. Ensure your Accelerator is set to **GPU T4 x2** or **GPU P100**.

In [ ]:
import os
import sys

# 1. Define Paths for Kaggle Environment
# Kaggle mounts datasets read-only at /kaggle/input/
# You MUST change the 'DATASET_NAME' below to match exactly what your dataset is called on Kaggle!
DATASET_NAME = "picai-pre-processed-dataset" 
SOURCE_DATA_DIR = f"/kaggle/input/{DATASET_NAME}"

# Kaggle provides a fast NVMe working directory at /kaggle/working/ for read-write
WORKSPACE_DIR = "/kaggle/working/PI-CAI_Workspace/baseline"

# We will save our model checkpoints here as well, so you can download them or save to Kaggle Output.
RESULTS_FOLDER = "/kaggle/working/PI-CAI_Results"

# 2. Domain Adaptation Setup
# ------------------------------------------------------------------
# We have 3 centers in the marksheet: RUMC, ZGT, PCNN.
# Set this to the 2 centers you want to train on (comma separated).
# The model will NEVER see cases from the 3rd center during training or validation.
TRAIN_CENTERS = "RUMC,ZGT"

# Optional testing limits (leave empty for full training run)
MAX_CASES = ""     
MAX_EPOCHS = ""    

# Export variables to environment
os.environ["SOURCE_DATA_DIR"] = SOURCE_DATA_DIR
os.environ["RESULTS_FOLDER"] = RESULTS_FOLDER
os.environ["WORKSPACE_DIR"] = WORKSPACE_DIR
os.environ["TRAIN_CENTERS"] = TRAIN_CENTERS
os.environ["MAX_CASES"] = MAX_CASES
os.environ["MAX_EPOCHS"] = MAX_EPOCHS

print("Environment configured for Kaggle!")
print(f"Training on centers: {TRAIN_CENTERS}")

In [ ]:
# 3. Clone repositories to fast local disk
!mkdir -p /kaggle/working/PI-CAI_Workspace

import os
if not os.path.exists("/kaggle/working/PI-CAI_Workspace/baseline"):
    !cd /kaggle/working/PI-CAI_Workspace && git clone https://github.com/HemishJain09/PI-CAI-Baseline.git baseline
else:
    !cd /kaggle/working/PI-CAI_Workspace/baseline && git pull

if not os.path.exists("/kaggle/working/PI-CAI_Workspace/Z-SSMNet"):
    !cd /kaggle/working/PI-CAI_Workspace && git clone https://github.com/yuanyuan29/Z-SSMNet.git Z-SSMNet

print("Repositories ready.")

In [ ]:
# 4. Install dependencies
import os
os.environ["SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL"] = "True"
!pip install -q -r $WORKSPACE_DIR/requirements.txt
!pip install -q git+https://github.com/DIAGNijmegen/nnUNet.git@1.7.0-3
print("Dependencies installed.")

In [ ]:
# 5. Run the full Domain Adaptation Pipeline
import os
os.chdir(os.environ["WORKSPACE_DIR"])
os.environ["SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL"] = "True"

# Execute pipeline
!chmod +x run_nnunet_pipeline.sh
!./run_nnunet_pipeline.sh